# V1 CUDA Test Notebook

Notebook này dùng để clone repository, build V0 CPU reference và kiểm tra các stage CUDA hiện có của Version 1 trên Google Colab.

Các kiểm tra gồm:
- V0 end-to-end panorama.
- S1–S2 CUDA parity test.
- S3–S4 CUDA parity test.
- S6 CUDA verification test.
- Compute Sanitizer nếu môi trường có sẵn.
- Timing harness S1–S2 của V0.

> Trước khi chạy, chọn **Runtime → Change runtime type → T4 GPU** trong Google Colab.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/nganbee/panorama-stitching-project.git'
BRANCH = 'v1'
REPO_DIR = Path('/content/panorama-stitching-project')

def run(command, check=True):
    print(f'$ {command}')
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')
    return result.returncode

if not REPO_DIR.exists():
    run(f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
else:
    run(f'git -C {REPO_DIR} fetch origin {BRANCH}')
    run(f'git -C {REPO_DIR} checkout {BRANCH}')
    run(f'git -C {REPO_DIR} pull --ff-only origin {BRANCH}', check=False)

os.chdir(REPO_DIR)
Path('build').mkdir(exist_ok=True)
Path('out').mkdir(exist_ok=True)
Path('reports').mkdir(exist_ok=True)
print('Working directory:', Path.cwd())

## 1. Kiểm tra GPU và CUDA

In [ ]:
run('nvidia-smi')
run('nvcc --version')
print('nvcc:', shutil.which('nvcc'))
print('compute-sanitizer:', shutil.which('compute-sanitizer'))

## 2. Build và chạy V0 CPU reference

V0 được dùng làm baseline để kiểm tra kết quả và timing.

In [ ]:
run('g++ -O2 -std=c++17 -Isrc/core_headers -Isrc/libraries '
    'src/integration/main.cpp src/integration/v0_pipeline.cpp '
    'src/stage_implementations/v0_stage12.cpp '
    'src/stage_implementations/v0_stage34.cpp '
    'src/stage_implementations/v0_stage5.cpp '
    'src/stage_implementations/v0_stage6.cpp '
    'src/libraries/stb_impl.cpp -o build/stitch')

In [ ]:
LEFT = 'dataset/location1/small_720p/1.jpg'
RIGHT = 'dataset/location1/small_720p/2.jpg'
V0_OUTPUT = 'out/panorama_v0_location1_720p.png'
V0_JSON = 'reports/v0_location1_720p.json'
run(f'build/stitch {LEFT} {RIGHT} {V0_OUTPUT} --version 0 --runs 5 --json {V0_JSON}')

In [ ]:
from IPython.display import Image, display
display(Image(filename=V0_OUTPUT))

## 3. Self-test S1–S2 CUDA

Test này so sánh grayscale, Gaussian blur, Harris response, NMS và top-K của V1 với V0.

In [ ]:
run('nvcc -O2 -std=c++17 -Isrc/core_headers '
    'tools/run_stage12_cuda_selftest.cpp '
    'src/stage_implementations/v0_stage12.cpp '
    'src/stage_implementations/v1_stage12.cu '
    '-o build/stage12_cuda_selftest')
run('build/stage12_cuda_selftest')

In [ ]:
if shutil.which('compute-sanitizer'):
    run('compute-sanitizer build/stage12_cuda_selftest')
else:
    print('compute-sanitizer is not available; skipped.')

## 4. Self-test S3–S4 CUDA

In [ ]:
run('nvcc -O2 -std=c++17 -Isrc/core_headers '
    'tools/run_stage34_cuda_selftest.cpp '
    'src/stage_implementations/v0_stage34.cpp '
    'src/stage_implementations/v1_stage34.cu '
    '-o build/stage34_cuda_selftest')
run('build/stage34_cuda_selftest')

## 5. Self-test S6 CUDA

In [ ]:
run('nvcc -O2 -std=c++17 -Isrc/core_headers '
    'tools/test_v1_stage6.cpp '
    'src/stage_implementations/v0_stage6.cpp '
    'src/stage_implementations/v1_stage6.cu '
    '-o build/test_v1_stage6')
run('build/test_v1_stage6')

## 6. Timing harness S1–S2 của V0

Harness có một warm-up và trung bình tối thiểu năm lần chạy ở 720p, 1080p và 4K.

In [ ]:
run('g++ -O2 -std=c++17 -Isrc/core_headers -Isrc/libraries '
    'tools/timing_stage12.cpp '
    'src/stage_implementations/v0_stage12.cpp '
    'src/libraries/stb_impl.cpp '
    '-o build/timing_stage12')
run(f'build/timing_stage12 5 {LEFT} {RIGHT}')

## 7. Build và chạy full V1 pipeline

V1 dùng CUDA cho S1–S4 và S6; S5 vẫn dùng implementation CPU V0 trong Milestone 2.

In [ ]:
run('nvcc -O2 -std=c++17 -DPANORAMA_WITH_CUDA '
    '-Isrc/core_headers -Isrc/libraries '
    'src/integration/main.cpp src/integration/v0_pipeline.cpp '
    'src/stage_implementations/v0_stage12.cpp '
    'src/stage_implementations/v0_stage34.cpp '
    'src/stage_implementations/v0_stage5.cpp '
    'src/stage_implementations/v0_stage6.cpp '
    'src/stage_implementations/v1_stage12.cu '
    'src/stage_implementations/v1_stage34.cu '
    'src/stage_implementations/v1_stage6.cu '
    'src/cuda/cuda_pipeline.cu src/libraries/stb_impl.cpp '
    '-o build/stitch_cuda')

In [ ]:
V1_OUTPUT = 'out/panorama_v1_location1_720p.png'
V1_JSON = 'reports/v1_location1_720p.json'
run(f'build/stitch_cuda {LEFT} {RIGHT} {V1_OUTPUT} --version 1 --runs 5 --json {V1_JSON}')
run('build/stitch_cuda --selftest --version 1')

In [ ]:
from PIL import Image as PILImage
import numpy as np

v0 = np.asarray(PILImage.open(V0_OUTPUT).convert('RGB')).astype(np.int16)
v1 = np.asarray(PILImage.open(V1_OUTPUT).convert('RGB')).astype(np.int16)
print('V0 shape:', v0.shape)
print('V1 shape:', v1.shape)
if v0.shape == v1.shape:
    diff = np.abs(v0 - v1)
    print('mean_abs_pixel_error:', float(diff.mean()))
    print('max_abs_pixel_error:', int(diff.max()))
else:
    print('Panorama dimensions differ; compare homography/crop diagnostics before pixel error.')
display(Image(filename=V1_OUTPUT))

## 8. Kiểm tra trạng thái full V1 pipeline

In [ ]:
cuda_pipeline = Path('src/cuda/cuda_pipeline.cu')
print('Full CUDA pipeline source:', cuda_pipeline)
print('V1 executable:', Path('build/stitch_cuda'))
print('V1 report:', Path(V1_JSON))

## 9. Kết quả mong đợi

Các cell cần lần lượt đạt:

```text
S1/S2 CUDA parity tests passed
S3/S4 CUDA parity tests passed
Result: ALL PASS
V1 CUDA S1-S2 smoke selftest PASS
version=1 ...
```

Nếu một parity test hoặc full pipeline thất bại, giữ lại toàn bộ output của cell, thông tin GPU từ `nvidia-smi`, phiên bản CUDA từ `nvcc --version`, và dataset đang dùng để debug.